# HDS-M03 Mini Project: Lung Segmentation from Chest X-rays

## Part 1: Preparing the Imaging Data (steps 1–3)

In this notebook I check that the Montgomery data loads correctly, look at what an X-ray and its
ground-truth mask look like, and resample the images to a manageable size.

All repeated code is placed in `lungseg.py`, which is imported below.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# local helper module, all repeated code lives in lungseg.py
import lungseg

# create a folder for figures if it doesn't exist
os.makedirs('figures', exist_ok=True)

#### (1) Choosing a subset of the data

To begin, I check that the data can be found and count the images in each class.
If the counts are 0, the `DATA_DIR` line at the top of `lungseg.py` needs changing (see `data/README.md`).

In [ ]:
print(f'Looking for images in: {lungseg.IMAGE_DIR.resolve()}')

# list file names for all cases, normal cases (_0.png) and TB cases (_1.png)
all_cases    = lungseg.list_cases()
normal_cases = lungseg.list_cases(label=0)
tb_cases     = lungseg.list_cases(label=1)

print(f'Total images : {len(all_cases)}')
print(f'Normal       : {len(normal_cases)}')
print(f'TB           : {len(tb_cases)}')
print(f'First few    : {all_cases[:3]}')

#### (2) Loading and visualising the X-rays and ground truth

Before any processing, I check the raw size and data type of one X-ray.

In [ ]:
from skimage import io

# read the first normal X-ray exactly as stored on disk
first = normal_cases[0]
raw   = io.imread(lungseg.IMAGE_DIR / first)

print(f'File        : {first}')
print(f'Shape       : {raw.shape}')
print(f'Dtype       : {raw.dtype}')
print(f'Value range : [{raw.min()}, {raw.max()}]')
print(f'Megapixels  : {raw.size / 1e6:.1f}')

Next I visualise a small subset (3 normal and 3 TB cases), showing the X-ray, the radiologist's
mask, and the mask outline drawn on the X-ray.

In [ ]:
# 3 normal + 3 TB file names
examples = normal_cases[:3] + tb_cases[:3]

fig, axes = plt.subplots(len(examples), 3, figsize=(10, 3.3 * len(examples)))

for row, name in enumerate(examples):

    # load the resized X-ray and combined left + right lung mask
    img, gt = lungseg.load_case(name)
    label   = 'TB' if lungseg.get_label(name) else 'normal'

    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'{name}\n({label})')

    axes[row, 1].imshow(gt, cmap='gray')
    axes[row, 1].set_title('Ground truth mask')

    # draw the mask outline in red on top of the X-ray
    axes[row, 2].imshow(img, cmap='gray')
    axes[row, 2].contour(gt, levels=[0.5], colors='r')
    axes[row, 2].set_title('Overlay')

# hide axes for a cleaner look
for ax in axes.ravel():
    ax.axis('off')

plt.tight_layout()
plt.savefig('figures/fig_1_examples.pdf', bbox_inches='tight')
plt.show()

From these images:

- Lungs are **dark** (air absorbs few X-rays), while bone and soft tissue are **bright**.
- The area *outside* the body is also dark, so a simple "dark = lung" rule will pick that up too.
- Ground-truth masks usually **exclude** the heart and the region behind it.
- TB cases can have bright patches *inside* the lung, which may confuse a threshold.

#### (3) Resampling the images

The originals are millions of pixels, which makes filtering, morphology and GLCM slow, and lung
boundaries don't need that much detail. `lungseg.load_case` therefore resizes every image to
`lungseg.IMAGE_SIZE` (512 × 512):

- the **X-ray** is resized with `anti_aliasing=True`, which blurs slightly first to avoid jagged artefacts;
- the **mask** is resized with nearest neighbour (`order=0`), so it stays strictly True/False;
- the left and right lung masks are combined into **one** lung mask.

In [ ]:
image, mask = lungseg.load_case(first)

print(f'Resized image : {image.shape}, {image.dtype}, range [{image.min():.2f}, {image.max():.2f}]')
print(f'Resized mask  : {mask.shape}, {mask.dtype}, {mask.sum()} lung pixels')